# 05 Diffusion Innovation Model

This notebook trains a simple DDPM-style diffusion model on VAR residuals.

Goal:

Fit a flexible innovation generator to residuals and compare generated diffusion shocks against empirical residuals and classical innovation baselines.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

from src.utils.seeds import set_seed
from src.utils.device import get_device

from src.experiments.config import ExperimentConfig
from src.experiments.paths import result_dirs
from src.experiments.artifacts import (
    load_array_npz,
    save_array_npz,
    save_table,
    save_json,
    save_current_figure,
)

from src.api.innovations import (
    fit_innovations,
    sample_innovations,
)

from src.innovations.diagnostics import summarize_innovations

set_seed(123)

config = ExperimentConfig()
device = get_device()

dirs = result_dirs(
    "05_diffusion",
    test=False,
)

device

In [ ]:
forecast_models = [
    "VAR",
    "RNN",
]

dgp_names = config.dgp_names

diffusion_kwargs = {
    "timesteps": 200,
    "beta_start": 1e-4,
    "beta_end": 2e-2,
    "batch_size": 64,
    "epochs": 1500,
    "lr": 5e-4,
    "hidden_dim": 256,
    "time_embedding_dim": 64,
    "n_samples": 10_000,
    "device": device,
}

resid_dir = ROOT / "results" / "residuals" / "03_residuals"

residual_store = {}

for forecast_model in forecast_models:

    residual_store[forecast_model] = {}

    model_key = forecast_model.lower()

    for name in dgp_names:
        path = resid_dir / f"{name}_{model_key}_residuals.npz"
        data = load_array_npz(path)

        residuals = data["residuals"]

        residual_store[forecast_model][name] = residuals

        print(
            forecast_model,
            name,
            residuals.shape,
        )

In [ ]:
diffusion_models = {}
diffusion_samples = {}

for forecast_model in forecast_models:

    diffusion_models[forecast_model] = {}
    diffusion_samples[forecast_model] = {}

    for dgp_name in dgp_names:

        residuals = residual_store[
            forecast_model
        ][
            dgp_name
        ]

        print(
            f"\nTraining diffusion model:"
            f" {forecast_model} | {dgp_name}"
        )

        diffusion_model = fit_innovations(
            residuals=residuals,
            method="diffusion",
            save=True,
            output_dir=(
                dirs["innovations"]
                / forecast_model.lower()
                / dgp_name
                / "diffusion"
            ),
            **diffusion_kwargs,
        )

        samples = sample_innovations(
            innovation_model=diffusion_model,
            n_paths=10_000,
            horizon=1,
            seed=config.seed,
        )

        samples = samples[:, 0, :]

        diffusion_models[
            forecast_model
        ][
            dgp_name
        ] = diffusion_model

        diffusion_samples[
            forecast_model
        ][
            dgp_name
        ] = samples

        save_array_npz(
            (
                dirs["innovations"]
                / forecast_model.lower()
                / dgp_name
                / "diffusion_samples.npz"
            ),
            innovations=samples,
        )

        print(
            forecast_model,
            dgp_name,
            samples.shape,
        )

In [ ]:
fig, axes = plt.subplots(
    2,
    len(dgp_names),
    figsize=(16, 7),
    sharey=True,
)

for row, forecast_model in enumerate(forecast_models):
    for col, dgp_name in enumerate(dgp_names):

        losses = diffusion_models[
            forecast_model
        ][
            dgp_name
        ]["history"]["loss"]

        ax = axes[row, col]

        ax.plot(losses)

        ax.set_title(
            f"{forecast_model} | {dgp_name}"
        )

        ax.set_xlabel("epoch")

        if col == 0:
            ax.set_ylabel("loss")

plt.tight_layout()

save_current_figure(
    dirs["figures"]
    / "diffusion_training_losses.png"
)

plt.show()

In [ ]:
rows = []

for forecast_model in forecast_models:
    for dgp_name in dgp_names:
        empirical = residual_store[forecast_model][dgp_name]
        generated = diffusion_samples[forecast_model][dgp_name]

        emp_diag = summarize_innovations(empirical)
        gen_diag = summarize_innovations(generated)

        for j in range(empirical.shape[1]):
            rows.append({
                "forecast_model": forecast_model,
                "dgp": dgp_name,
                "series": j + 1,
                "source": "empirical_residuals",
                "mean": emp_diag["mean"][j],
                "std": emp_diag["std"][j],
                "skewness": emp_diag["skewness"][j],
                "kurtosis": emp_diag["kurtosis"][j],
                "excess_kurtosis": emp_diag["excess_kurtosis"][j],
                "jb_pvalue": emp_diag["jarque_bera_pvalue"][j],
            })

            rows.append({
                "forecast_model": forecast_model,
                "dgp": dgp_name,
                "series": j + 1,
                "source": "diffusion",
                "mean": gen_diag["mean"][j],
                "std": gen_diag["std"][j],
                "skewness": gen_diag["skewness"][j],
                "kurtosis": gen_diag["kurtosis"][j],
                "excess_kurtosis": gen_diag["excess_kurtosis"][j],
                "jb_pvalue": gen_diag["jarque_bera_pvalue"][j],
            })

diffusion_diag_df = pd.DataFrame(rows)

save_table(
    diffusion_diag_df,
    dirs["tables"] / "diffusion_diagnostics_by_series.csv",
)

diffusion_diag_df

In [ ]:
summary_df = (
    diffusion_diag_df
    .groupby(
        [
            "forecast_model",
            "dgp",
            "source",
        ]
    )[
        [
            "mean",
            "std",
            "skewness",
            "kurtosis",
            "excess_kurtosis",
        ]
    ]
    .mean()
    .reset_index()
)

save_table(
    summary_df,
    dirs["tables"] / "diffusion_diagnostics_summary.csv",
)

summary_df

In [ ]:
fig, axes = plt.subplots(
    4,
    len(dgp_names),
    figsize=(16, 14),
    sharex=False,
    sharey=False,
)

row_specs = [
    ("VAR", "Empirical"),
    ("VAR", "Diffusion"),
    ("RNN", "Empirical"),
    ("RNN", "Diffusion"),
]

for row, (forecast_model, source) in enumerate(row_specs):
    for col, dgp_name in enumerate(dgp_names):

        if source == "Empirical":
            x = residual_store[forecast_model][dgp_name][:, 0]
        else:
            x = diffusion_samples[forecast_model][dgp_name][:, 0]

        axes[row, col].hist(
            x,
            bins=35,
            density=True,
            alpha=0.7,
        )

        if row == 0:
            axes[row, col].set_title(dgp_name)

        if col == 0:
            axes[row, col].set_ylabel(
                f"{forecast_model}\n{source}"
            )

plt.tight_layout()

save_current_figure(
    dirs["figures"] / "diffusion_histogram_grid_4x4.png"
)

plt.show()

In [ ]:
fig, axes = plt.subplots(
    4,
    len(dgp_names),
    figsize=(16, 14),
)

row_specs = [
    ("VAR", "Empirical"),
    ("VAR", "Diffusion"),
    ("RNN", "Empirical"),
    ("RNN", "Diffusion"),
]

for row, (forecast_model, source) in enumerate(row_specs):
    for col, dgp_name in enumerate(dgp_names):

        if source == "Empirical":
            x = residual_store[forecast_model][dgp_name][:, 0]
        else:
            x = diffusion_samples[forecast_model][dgp_name][:, 0]

        stats.probplot(
            x,
            dist="norm",
            plot=axes[row, col],
        )

        if row == 0:
            axes[row, col].set_title(dgp_name)

        if col == 0:
            axes[row, col].set_ylabel(
                f"{forecast_model}\n{source}"
            )

plt.tight_layout()

save_current_figure(
    dirs["figures"] / "diffusion_qq_grid_4x4.png"
)

plt.show()

In [ ]:
quantiles = [0.01, 0.05, 0.50, 0.95, 0.99]

q_rows = []

for forecast_model in forecast_models:
    for dgp_name in dgp_names:

        empirical = residual_store[
            forecast_model
        ][
            dgp_name
        ]

        generated = diffusion_samples[
            forecast_model
        ][
            dgp_name
        ]

        for j in range(empirical.shape[1]):

            emp_q = np.quantile(
                empirical[:, j],
                quantiles,
            )

            gen_q = np.quantile(
                generated[:, j],
                quantiles,
            )

            for q, e, g in zip(
                quantiles,
                emp_q,
                gen_q,
            ):
                q_rows.append(
                    {
                        "forecast_model": forecast_model,
                        "dgp": dgp_name,
                        "series": j + 1,
                        "quantile": q,
                        "empirical": e,
                        "diffusion": g,
                        "absolute_error": abs(e - g),
                    }
                )

quantile_df = pd.DataFrame(q_rows)

save_table(
    quantile_df,
    dirs["tables"] / "diffusion_quantile_recovery.csv",
)

quantile_df

In [ ]:
quantile_summary_df = (
    quantile_df
    .groupby(
        [
            "forecast_model",
            "dgp",
            "quantile",
        ]
    )["absolute_error"]
    .mean()
    .reset_index()
)

save_table(
    quantile_summary_df,
    dirs["tables"] / "diffusion_quantile_recovery_summary.csv",
)

quantile_summary_df

In [ ]:
moment_rows = []

for forecast_model in forecast_models:
    for dgp_name in dgp_names:

        empirical = residual_store[
            forecast_model
        ][
            dgp_name
        ]

        generated = diffusion_samples[
            forecast_model
        ][
            dgp_name
        ]

        emp_mean = empirical.mean(axis=0)
        gen_mean = generated.mean(axis=0)

        emp_cov = np.cov(
            empirical,
            rowvar=False,
        )

        gen_cov = np.cov(
            generated,
            rowvar=False,
        )

        emp_skew = stats.skew(
            empirical,
            axis=0,
        )

        gen_skew = stats.skew(
            generated,
            axis=0,
        )

        emp_kurt = stats.kurtosis(
            empirical,
            axis=0,
            fisher=False,
        )

        gen_kurt = stats.kurtosis(
            generated,
            axis=0,
            fisher=False,
        )

        moment_rows.append(
            {
                "forecast_model": forecast_model,
                "dgp": dgp_name,
                "mean_l2_error":
                    np.linalg.norm(
                        emp_mean - gen_mean
                    ),
                "cov_frobenius_error":
                    np.linalg.norm(
                        emp_cov - gen_cov,
                        ord="fro",
                    ),
                "skew_l2_error":
                    np.linalg.norm(
                        emp_skew - gen_skew
                    ),
                "kurtosis_l2_error":
                    np.linalg.norm(
                        emp_kurt - gen_kurt
                    ),
            }
        )

moment_error_df = pd.DataFrame(moment_rows)

save_table(
    moment_error_df,
    dirs["tables"] / "diffusion_moment_errors.csv",
)

moment_error_df

In [ ]:
save_json(
    {
        "notebook": "05_diffusion",
        "forecast_models": forecast_models,
        "dgp_names": dgp_names,
        "diffusion_kwargs": {
            key: str(value)
            if "device" in key
            else value
            for key, value in diffusion_kwargs.items()
        },
        "models_saved": True,
        "samples_saved": True,
    },
    dirs["logs"] / "diffusion_config.json",
)

print(
    "05_diffusion completed."
)